<a href="https://colab.research.google.com/github/LGLV-Ciencia-de-Datos/Curso_python_Ciencia_de_Datos/blob/main/d)_Pipelines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Tuberías** (Pipelines)
Una habilidad fundamental para desplegar (e incluso probar) modelos complejos con preprocesamiento

En este tutorial, aprenderás cómo usar pipelines para organizar tu código de modelado.

### **Introducción**
Los pipelines son una forma sencilla de mantener organizado tu código de preprocesamiento de datos y modelado. Específicamente, un pipeline agrupa los pasos de preprocesamiento y modelado para que puedas usar todo el conjunto como si fuera un solo paso.
Muchos científicos de datos montan sus modelos sin pipelines, pero estos tienen ventajas importantes. Entre ellas se incluyen:
1. **Código más limpio**: Tener en cuenta los datos en cada paso de preprocesamiento puede complicarse. Con un pipeline, no será necesario hacer un seguimiento manual de los datos de entrenamiento y validación en cada etapa.
2. **Menos errores**: Hay menos oportunidades de aplicar incorrectamente un paso o de olvidar algún paso de preprocesamiento.
3. **Más fácil de poner en producción**: Puede ser sorprendentemente difícil pasar de un prototipo a algo desplegable a escala. No entraremos en todos los aspectos relacionados aquí, pero los pipelines pueden ayudar.
4. **Más opciones para la validación del modelo**: Verás un ejemplo en el próximo tutorial, que trata sobre validación cruzada.

### **Ejemplo**
Al igual que en el tutorial anterior, trabajaremos con el conjunto de datos de viviendas de Melbourne.
No nos centraremos en la fase de carga de datos. En su lugar, puedes imaginar que ya tienes los datos de entrenamiento y validación en X_train, X_valid, y_train y y_valid.

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("dansbecker/melbourne-housing-snapshot")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os

# Get the data path from the previous cell's output
data_path = '/kaggle/input/melbourne-housing-snapshot/melb_data.csv'

# Read the data
data = pd.read_csv(data_path)

# Separate target from predictors
y = data.Price
X = data.drop(['Price'], axis=1)

# Divide data into training and validation subsets
X_train_full, X_valid_full, y_train, y_valid = train_test_split(X, y, train_size=0.8, test_size=0.2,
                                                                random_state=0)

# "Cardinality" means the number of unique values in a column
# Select categorical columns with relatively low cardinality (convenient but arbitrary)
categorical_cols = [cname for cname in X_train_full.columns if X_train_full[cname].nunique() < 10 and
                        X_train_full[cname].dtype == "object"]

# Select numerical columns
numerical_cols = [cname for cname in X_train_full.columns if X_train_full[cname].dtype in ['int64', 'float64']]

# Keep selected columns only
my_cols = categorical_cols + numerical_cols
X_train = X_train_full[my_cols].copy()
X_valid = X_valid_full[my_cols].copy()

print("Data loaded successfully!")

Echamos un vistazo a los datos de entrenamiento con el método `head()` a continuación. Observa que los datos contienen tanto datos categóricos como columnas con valores faltantes. ¡Con un pipeline, es fácil manejar ambos!

In [ ]:
X_train.head()

Construimos toda la canalización en tres pasos.
### **Paso 1:** Definir los pasos de preprocesamiento
De manera similar a cómo una canalización agrupa los pasos de preprocesamiento y modelado, utilizamos la clase `ColumnTransformer` para agrupar diferentes pasos de preprocesamiento. El código a continuación:
* imputa los valores faltantes en datos numéricos, y
* imputa los valores faltantes y aplica una codificación one-hot a los datos categóricos.

In [ ]:
from sklearn.compose import ColumnTransformer
# Se importa la clase ColumnTransformer, que permite aplicar transformaciones
# diferentes a subconjuntos de columnas de un conjunto de datos.

from sklearn.pipeline import Pipeline
# Se importa la clase Pipeline, que encadena varios pasos de preprocesamiento
# y modelado en un solo objeto. Esto es útil para mantener el flujo de trabajo
# ordenado y evitar fugas de datos.

from sklearn.impute import SimpleImputer
# Se importa la clase SimpleImputer, que se usa para manejar los valores
# faltantes en un conjunto de datos, por ejemplo, rellenándolos con la
# media, la mediana o un valor constante.

from sklearn.preprocessing import OneHotEncoder
# Se importa la clase OneHotEncoder, que se utiliza para convertir variables
# categóricas nominales (como 'rojo', 'azul') en un formato numérico que
# los algoritmos de aprendizaje automático puedan entender. Crea una
# columna binaria por cada categoría.
################################################################################
# Preprocesamiento para datos numéricos
numerical_transformer = SimpleImputer(strategy='constant')
# Se crea una instancia de SimpleImputer para las columnas numéricas.
# La estrategia 'constant' reemplaza todos los valores faltantes (NaN)
# en las columnas numéricas con un valor constante, que por defecto es 0.

# Preprocesamiento para datos categóricos
categorical_transformer = Pipeline(steps=[
# Se crea un Pipeline para las columnas categóricas, lo que permite
# aplicar múltiples pasos de preprocesamiento en secuencia.

    ('imputer', SimpleImputer(strategy='most_frequent')),
# El primer paso del pipeline es un SimpleImputer. La estrategia
# 'most_frequent' reemplaza los valores faltantes con el valor más
# común en cada columna categórica.

    ('onehot', OneHotEncoder(handle_unknown='ignore'))
# El segundo paso es un OneHotEncoder. El parámetro `handle_unknown='ignore'`
# asegura que si una categoría nueva (no vista durante el entrenamiento)
# aparece en los datos de prueba, no se generará un error y su codificación
# será de ceros.
])
################################################################################
# Agrupar el preprocesamiento para datos numéricos y categóricos
preprocessor = ColumnTransformer(
# Se crea la instancia principal, ColumnTransformer, que aplicará los
# transformadores definidos a las columnas especificadas.
    transformers=[
# La lista `transformers` contiene tuplas, donde cada tupla define una
# transformación.

        ('num', numerical_transformer, numerical_cols),
# La primera tupla:
# 'num': Es un nombre descriptivo para esta transformación.
# `numerical_transformer`: El transformador a aplicar (el SimpleImputer).
# `numerical_cols`: Una lista o un array con los nombres de las columnas
# a las que se aplicará este transformador.

        ('cat', categorical_transformer, categorical_cols)
# La segunda tupla:
# 'cat': Un nombre descriptivo para la transformación de datos categóricos.
# `categorical_transformer`: El transformador a aplicar (el Pipeline).
# `categorical_cols`: Una lista o un array con los nombres de las columnas
# a las que se aplicará este transformador.
])
################################################################################

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Preprocesamiento para datos numéricos
numerical_transformer = SimpleImputer(strategy='constant')

# Preprocesamiento para datos categóricos
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Agrupar el preprocesamiento para datos numéricos y categóricos
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])

###**Paso 2:** Definir el modelo
A continuación, definimos un modelo de bosque aleatorio con la clase familiar `RandomForestRegressor`.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=100, random_state=0)


### **Paso 3:** Crear y evaluar la tubería (Pipeline)
Por último, utilizamos la clase [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) para definir una tubería que agrupa los pasos de preprocesamiento y modelado. Hay algunas cosas importantes a tener en cuenta:
* Con pipeline, preprocesamos los datos de entrenamiento y ajustamos el modelo en una sola línea de código. (En contraste, sin pipeline, tenemos que realizar la imputación, la codificación one-hot y el entrenamiento del modelo en pasos separados. ¡Esto resulta especialmente complicado si tenemos que tratar con variables tanto numéricas como categóricas!)
* Con pipeline, proporcionamos las características sin procesar en `X_valid` a la función `predict()`, y la tubería preprocesa automáticamente las características antes de generar las predicciones. (Sin embargo, sin una pipeline, debemos recordar preprocesar los datos de validación antes de hacer predicciones.)


In [ ]:
# Importa la función 'mean_absolute_error' para calcular el error absoluto medio.
# El MAE es una métrica de evaluación que mide la diferencia promedio entre los valores reales y los valores predichos.
from sklearn.metrics import mean_absolute_error

# Se crea una 'Pipeline' (tubería) que encadena los pasos de preprocesamiento y el modelo.
# Esto simplifica el flujo de trabajo y asegura que los mismos pasos se apliquen de manera consistente a los datos de entrenamiento y validación.
my_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                              ('model', model)
                             ])

# 'fit' entrena el modelo de la pipeline.
# Primero, el 'preprocessor' aprende las transformaciones necesarias de los datos de entrenamiento (X_train).
# Luego, el modelo se entrena con los datos preprocesados.
my_pipeline.fit(X_train, y_train)

# 'predict' realiza predicciones sobre los datos de validación.
# La pipeline primero aplica las transformaciones aprendidas del 'preprocessor' a los datos de validación (X_valid).
# Luego, el modelo entrenado genera las predicciones.
preds = my_pipeline.predict(X_valid)

# Se calcula el 'mean_absolute_error' (MAE) comparando los valores reales (y_valid) con las predicciones (preds).
score = mean_absolute_error(y_valid, preds)

# Imprime el valor del MAE, que indica la precisión del modelo en los datos de validación.
# Un valor más bajo de MAE significa que las predicciones del modelo están más cerca de los valores reales.
print('MAE:', score)

In [ ]:
from sklearn.metrics import mean_absolute_error

# Agrupamiento del código de preprocesamiento y modelado en un pipeline
my_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                              ('model', model)
                             ])

# Preprocesamiento de los datos de entrenamiento, se entrena el modelo
my_pipeline.fit(X_train, y_train)

# Preprocesamiento de los datos de validación, para obtener predicciones
preds = my_pipeline.predict(X_valid)

# Evaluación del modelo
score = mean_absolute_error(y_valid, preds)
print('MAE:', score)

### **Conclusión**
Los procesos son valiosos para limpiar el código de aprendizaje automático y evitar errores, y son especialmente útiles en flujos de trabajo con un preprocesamiento de datos sofisticado.